In [1]:
from census import Census
from pygris import block_groups
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [2]:
# =====================================================
# CONFIG
# =====================================================

API_KEY = open("../../data/census/api.txt").read().strip()

STATE_FIPS = "06"
COUNTY_FIPS = "001"

In [3]:
# =====================================================
# ACS DATA
# =====================================================

c = Census(API_KEY, year=2024)

# Define your variables: code -> friendly column name
ACS_VARIABLES = {
    "B01003_001E": "pop_count",
    "B19013_001E": "median_household_income",
    "B25077_001E": "median_home_value",
    # --- new corridor-relevant variables added below ---
    "B11001_001E": "total_households",        # for purchasing-power index

    # Commuting / foot-traffic & parking-demand proxies (B08301)
    "B08301_001E": "commute_total",
    "B08301_003E": "drove_alone",
    "B08301_009E": "public_transit",
    "B08301_017E": "bicycle",
    "B08301_018E": "walked",

    # Vehicle access (B08201)
    "B08201_001E": "hh_vehicle_universe",
    "B08201_002E": "zero_vehicle_hh",

    # Tenure / displacement-risk exposure (B25003)
    "B25003_001E": "tenure_total",
    "B25003_003E": "renter_occupied",

    # Nativity / immigrant-entrepreneurship proxy (B05001)
    "B05001_001E": "citizenship_universe",
    "B05001_005E": "naturalized_citizen",
    "B05001_006E": "not_us_citizen",

    # Household income distribution, low-income brackets (B19001)
    "B19001_001E": "income_bracket_universe",
    "B19001_002E": "income_under_10k",
    "B19001_003E": "income_10_15k",
    "B19001_004E": "income_15_20k",
    "B19001_005E": "income_20_25k",
    "B19001_006E": "income_25_30k",
    "B19001_007E": "income_30_35k",

    # Industry of employed residents - retail/food-service tie to corridor (C24050)
    "C24050_001E": "employed_total",
    "C24050_006E": "retail_trade_employed",
    "C24050_012E": "arts_ent_accom_food_employed",

    # --- additional raw variables (context layers) ---
    "B25064_001E": "median_gross_rent",
    "B19301_001E": "per_capita_income",
    "B25010_001E": "avg_household_size",
    "B01002_001E": "median_age",
    "B25035_001E": "median_year_built",
    "B25002_001E": "total_housing_units",
}

print("Downloading ACS data...")

acs = c.acs5.state_county_blockgroup(
    tuple(ACS_VARIABLES.keys()),
    STATE_FIPS,
    COUNTY_FIPS,
    Census.ALL
)

df = pd.DataFrame(acs)

# Build GEOID
df["tract"] = df["tract"].str.zfill(6)
df["block group"] = df["block group"].str.zfill(1)
df["GEOID"] = (
    df["state"]
    + df["county"]
    + df["tract"]
    + df["block group"]
)

# Convert each variable to numeric and rename
for code, col_name in ACS_VARIABLES.items():
    df[col_name] = pd.to_numeric(df[code], errors="coerce")

# Drop raw census code columns, keep only friendly names + GEOID
raw_cols = list(ACS_VARIABLES.keys()) + ["state", "county", "tract", "block group"]
df = df.drop(columns=raw_cols)

# =====================================================
# BLOCK GROUP GEOMETRIES
# =====================================================

print("Downloading block group geometries...")

bg = block_groups(
    state="CA",
    county="Alameda",
    year=2024
)

# =====================================================
# JOIN ACS DATA
# =====================================================

acs_cols = ["GEOID"] + list(ACS_VARIABLES.values())

gdf = bg.merge(
    df[acs_cols],
    on="GEOID",
    how="left"
)

cols_to_drop = [
    "STATEFP", "COUNTYFP", "TRACTCE", "BLKGRPCE",
    "GEOIDFQ", "NAMELSAD", "MTFCC", "FUNCSTAT",
    "ALAND", "AWATER", "INTPTLON", "INTPTLAT"
]

gdf = gdf.drop(columns=cols_to_drop)

gdf.head()

Using FIPS code '06' for input 'CA'
Using FIPS code '001' for input 'Alameda'


,GEOID,geometry,pop_count,median_household_income,median_home_value,total_households,commute_total,drove_alone,public_transit,bicycle,...,income_30_35k,employed_total,retail_trade_employed,arts_ent_accom_food_employed,median_gross_rent,per_capita_income,avg_household_size,median_age,median_year_built,total_housing_units
0,060014423012,"POLYGON ((-121.96678 37.5303, -121.96678 37.53...",1173.0,102617.0,1326900.0,425.0,616.0,410.0,0.0,0.0,...,0.0,NaN,NaN,NaN,2454.0,39663.0,2.76,41.2,1976,497.0
1,060014060001,"POLYGON ((-122.26838 37.78803, -122.26736 37.7...",2035.0,65425.0,784200.0,1058.0,1151.0,541.0,0.0,0.0,...,0.0,NaN,NaN,NaN,2116.0,52351.0,1.88,39.4,2014,1131.0
2,060014337001,"POLYGON ((-122.11467 37.6887, -122.11462 37.68...",1508.0,91875.0,786400.0,419.0,817.0,519.0,0.0,0.0,...,29.0,NaN,NaN,NaN,1993.0,30208.0,3.58,33.4,1960,445.0
3,060014011004,"POLYGON ((-122.26764 37.82783, -122.2676 37.82...",2675.0,98570.0,781300.0,1215.0,1324.0,423.0,0.0,0.0,...,17.0,NaN,NaN,NaN,2602.0,69220.0,2.17,32.2,2013,1231.0
4,060014012001,"POLYGON ((-122.26016 37.83119, -122.26003 37.8...",1443.0,235109.0,1344400.0,686.0,927.0,222.0,0.0,0.0,...,0.0,NaN,NaN,NaN,2802.0,113397.0,2.10,38.0,1958,686.0


In [4]:
# =====================================================
# DENSITY CALCULATION
# =====================================================

gdf = gdf.to_crs(3310)

gdf["area_sqkm"] = (
    gdf.geometry.area / 1_000_000
)

gdf["pop_density"] = (
    gdf["pop_count"] /
    gdf["area_sqkm"]
)

gdf = gdf.to_crs(4326)

In [5]:
# =====================================================
# VARIABLE 1: DER-POP-01 - Population density (per sq. mi.)
# =====================================================
# Foot-traffic / walkability context - area_sqkm computed in the DENSITY cell above.

SQKM_TO_SQMI = 0.386102

gdf["area_sqmi"] = gdf["area_sqkm"] * SQKM_TO_SQMI
gdf["pop_density_sqmi"] = gdf["pop_count"] / gdf["area_sqmi"]

In [6]:
# =====================================================
# VARIABLE 2: DER-INC-05 - Aggregate purchasing power index
# =====================================================
# Market-sizing layer for retail siting / gap analysis.
# Formula: median household income x total households (by block group).

gdf["purchasing_power_index"] = gdf["median_household_income"] * gdf["total_households"]

In [7]:
# =====================================================
# VARIABLE 3: DER-INC-06 - Purchasing power density (per sq. mi.)
# =====================================================
# Highlights where corridor market demand is spatially densest.

gdf["purchasing_power_density"] = gdf["purchasing_power_index"] / gdf["area_sqmi"]

In [8]:
# =====================================================
# VARIABLE 4: DER-TRAN-01 - Drive-alone commute share
# =====================================================
# Baseline auto-dependency measure, useful for parking-demand planning near the corridor.
# Formula: B08301_003 (drove alone) / B08301_001 (total workers 16+).

gdf["drive_alone_share"] = gdf["drove_alone"] / gdf["commute_total"]

In [9]:
# =====================================================
# VARIABLE 5: DER-TRAN-02 - Active/transit commute share (walk + bike + transit)
# =====================================================
# Strong proxy for street-level foot traffic and storefront visibility.
# NOTE: verify B08301 sub-cell numbers (009/017/018) against the Census variables
# list for your ACS vintage - the table has ~20 categories and numbering can shift.

gdf["active_transit_share"] = (
    gdf["public_transit"] + gdf["bicycle"] + gdf["walked"]
) / gdf["commute_total"]

In [10]:
# =====================================================
# VARIABLE 6: DER-TRAN-05 - Zero-vehicle household rate
# =====================================================
# Households that depend heavily on transit/walking, and thus on nearby retail.
# Formula: B08201_002 (no vehicle available) / B08201_001 (total households).

gdf["zero_vehicle_rate"] = gdf["zero_vehicle_hh"] / gdf["hh_vehicle_universe"]

In [11]:
# =====================================================
# VARIABLE 7: DER-HSG-01 - Renter share
# =====================================================
# Renters are typically more exposed to displacement from corridor investment.
# Formula: B25003_003 (renter-occupied) / B25003_001 (total occupied housing units).

gdf["renter_share"] = gdf["renter_occupied"] / gdf["tenure_total"]

In [12]:
# =====================================================
# VARIABLE 8: DER-LANG-01 - Foreign-born share
# =====================================================
# Proxy for immigrant entrepreneurship and cultural business districts along the corridor.
# Formula: foreign-born population / total population.
# Substituted B05001 (Citizenship Status) for the sheet's B05002 reference - simpler,
# equivalent totals (naturalized-citizen + not-a-citizen = foreign born).

gdf["foreign_born_share"] = (
    gdf["naturalized_citizen"] + gdf["not_us_citizen"]
) / gdf["citizenship_universe"]

In [13]:
# =====================================================
# VARIABLE 9: DER-INC-01 - Low-income household share (<$35,000)
# =====================================================
# Identifies concentrations of economically vulnerable households - relevant to
# retail mix and affordability along the corridor.
# Formula: sum of B19001 brackets under $35,000 / B19001_001 (total households).

gdf["low_income_share"] = (
    gdf["income_under_10k"] + gdf["income_10_15k"] + gdf["income_15_20k"]
    + gdf["income_20_25k"] + gdf["income_25_30k"] + gdf["income_30_35k"]
) / gdf["income_bracket_universe"]

In [14]:
# =====================================================
# VARIABLE 10: DER-EMP-05 - Retail & food-service employment share
# =====================================================
# Direct proxy for how much of the local workforce is already tied to
# corridor-type industries.
# NOTE: C24050's broad industry categories bundle "Arts, entertainment, recreation,
# and accommodation and food services" into one cell (_012) - food service can't be
# isolated from arts/entertainment at this level. Verify cell numbers/vintage before
# publishing; use a more detailed industry table if you need food service alone.

gdf["retail_food_service_share"] = (
    gdf["retail_trade_employed"] + gdf["arts_ent_accom_food_employed"]
) / gdf["employed_total"]

In [15]:
# =====================================================
# FILTER TO OAKLAND (whole block groups, by % overlap)
# =====================================================

oakland = gpd.read_file("../../data/geo/OaklandCityLimits/OaklandCityLimits.shp").to_crs(gdf.crs)

# Calculate what % of each block group falls within Oakland
gdf["bg_area"] = gdf.geometry.area
intersection = gpd.overlay(gdf, oakland[["geometry"]], how="intersection")
intersection["overlap_pct"] = intersection.geometry.area / intersection["bg_area"]

# Keep block groups with >50% overlap (adjust threshold as needed)
inside = intersection[intersection["overlap_pct"] > .5]["GEOID"]
gdf = gdf[gdf["GEOID"].isin(inside)].drop(columns="bg_area")

/var/folders/69/g590bg750s935v88zgfrdm9w0000gn/T/ipykernel_92366/686383270.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["bg_area"] = gdf.geometry.area
/var/folders/69/g590bg750s935v88zgfrdm9w0000gn/T/ipykernel_92366/686383270.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  intersection["overlap_pct"] = intersection.geometry.area / intersection["bg_area"]


In [16]:
# =====================================================
# RAW VARIABLE: Total households (B11001_001)
# =====================================================
# Kept as a standalone context layer (absolute market size), not just as the
# denominator for the purchasing-power index above. Just renaming for clarity.

gdf["total_households"] = gdf["total_households"]

In [17]:
# =====================================================
# RAW VARIABLE: Total employed population, 16+ (C24050_001)
# =====================================================
# Kept as a standalone workforce-base context layer alongside the
# retail/food-service employment share above.

gdf["total_employed"] = gdf["employed_total"]
gdf = gdf.drop(columns=["employed_total"])

In [18]:
# =====================================================
# RAW VARIABLE: Median gross rent (B25064_001)
# =====================================================
# Commercial-adjacent affordability signal; pairs with the purchasing-power layers.

gdf["median_gross_rent"] = gdf["median_gross_rent"].round(0)

In [19]:
# =====================================================
# RAW VARIABLE: Per capita income (B19301_001)
# =====================================================
# Complements median household income where household size varies widely
# across the corridor.

gdf["per_capita_income"] = gdf["per_capita_income"].round(0)

In [20]:
# =====================================================
# RAW VARIABLE: Average household size (B25010_001)
# =====================================================
# Used to translate population counts into household-level demand estimates.

gdf["avg_household_size"] = gdf["avg_household_size"].round(2)

In [21]:
# =====================================================
# RAW VARIABLE: Median age (B01002_001)
# =====================================================
# Signals whether the corridor's customer/labour base skews younger
# (nightlife, retail) or older (healthcare, personal services).

gdf["median_age"] = gdf["median_age"].round(1)

In [22]:
# =====================================================
# RAW VARIABLE: Median year structure built (B25035_001)
# =====================================================
# Proxy for building-stock age, relevant to storefront condition and
# redevelopment potential along the corridor.

gdf["median_year_built"] = gdf["median_year_built"].round(0)

In [23]:
# =====================================================
# RAW VARIABLE: Total housing units (B25002_001)
# =====================================================
# Context for residential density; also the denominator you'd use if you
# add a vacancy-rate layer later (B25002_003 / B25002_001).

gdf["total_housing_units"] = gdf["total_housing_units"]

In [24]:
# =====================================================
# CLEAN UP INTERMEDIATE COLUMNS
# =====================================================
# Drop the raw numerator/denominator columns now that the derived rates/indexes
# above have been calculated from them, keeping the friendly final metrics.

intermediate_cols = [
    "commute_total", "drove_alone", "public_transit", "bicycle", "walked",
    "hh_vehicle_universe", "zero_vehicle_hh",
    "tenure_total", "renter_occupied",
    "citizenship_universe", "naturalized_citizen", "not_us_citizen",
    "income_bracket_universe", "income_under_10k", "income_10_15k", "income_15_20k",
    "income_20_25k", "income_25_30k", "income_30_35k",
    "retail_trade_employed", "arts_ent_accom_food_employed",
    # NOTE: "total_households" and "employed_total" are kept (renamed below) as
    # useful raw context layers, not dropped as intermediates.
]

gdf = gdf.drop(columns=[c for c in intermediate_cols if c in gdf.columns])

In [25]:
# =====================================================
# SAVE
# =====================================================

gdf.to_file(
    "../../data/geo/block_groups/OAK_BG.geojson",
    driver="GeoJSON"
)

In [26]:
# =====================================================
# PRINT QUANTILES FOR VIZ FORMATTING
# =====================================================

numeric_cols = gdf.select_dtypes(include="number").columns.tolist()

rows = []
for col in numeric_cols:
    q1, q2, q3, q4 = gdf[col].quantile([0.2, 0.4, 0.6, 0.8])
    rows.append({"Column": col, "Q1 (20%)": q1, "Q2 (40%)": q2, "Q3 (60%)": q3, "Q4 (80%)": q4})

pd.DataFrame(rows).set_index("Column").round(2)

,Q1 (20%),Q2 (40%),Q3 (60%),Q4 (80%)
Column,,,,
pop_count,8.554000e+02,1.056000e+03,1.295800e+03,1.606200e+03
median_household_income,5.007200e+04,7.826940e+04,1.108718e+05,1.587096e+05
median_home_value,5.775000e+05,7.131200e+05,8.735600e+05,1.181720e+06
total_households,3.322000e+02,4.290000e+02,5.178000e+02,6.494000e+02
median_gross_rent,5.526000e+02,1.758000e+03,2.049200e+03,2.393000e+03
per_capita_income,2.882080e+04,4.427480e+04,6.827420e+04,9.585760e+04
avg_household_size,1.920000e+00,2.310000e+00,2.690000e+00,3.120000e+00
median_age,3.316000e+01,3.622000e+01,3.980000e+01,4.564000e+01
median_year_built,1.938000e+03,1.944000e+03,1.953000e+03,1.966400e+03
